In [1]:
from abc import ABC
import sys, os
import argparse

import xgboost as xgb
import pandas as pd
import numpy as np
from xgbt_train import build_X


In [2]:
# # モジュールの相対参照制限を強制的に回避
# current_dir = os.path.dirname(os.path.abspath(__file__))
# sys.path.append(os.path.join(current_dir, '..', 'analysis'))
# from xgbt_train import build_X

In [2]:
# TARGET = "stroke_flag"
TARGET = "depression_flag"
NUM_FEATURES = 21
NUM_CLASSES = 2

In [3]:
class Attack_Di_Base(ABC):
    def __init__(self, path_to_xgbt_model_json):
        """
        攻撃者の初期化

        path_to_xgbt_model_json: 学習済みのxgboostモデルのjsonファイルへのパス
        """
        # json fileを読み込み
        xgbt_model = xgb.Booster()
        xgbt_model.load_model(path_to_xgbt_model_json)

        self.xgbt_model = xgbt_model

        self.X = None
        self.y = None
        self.inferred = None
    
    def infer(self, path_to_Ai_csv):
        Ai_df = pd.read_csv(path_to_Ai_csv, dtype=str, 
                            keep_default_na=False)
        
        # 説明変数と目的変数に分割
        X = build_X(Ai_df, TARGET)

        # Xのみにある列は削除する, 9/10追記
        columns_only_X = set(X.columns) - set(self.xgbt_model.feature_names)
        if columns_only_X:
            X = X.drop(columns=columns_only_X)

        # xgbt_model.feature_namesのみにある列は0埋め, 9/10追記
        columns_only_feature_names = set(self.xgbt_model.feature_names) - set(X.columns)
        if columns_only_feature_names:
            for col in columns_only_feature_names:
                # 0で埋める
                X[col] = 0

        # Xの列をXGBoostモデルが要求する順番に並び替え, 9/10追記
        X = X.reindex(columns=self.xgbt_model.feature_names)

        X.columns = self.xgbt_model.feature_names
        self.X = X.copy()
        self.y = pd.to_numeric(Ai_df[TARGET], errors="coerce").astype(int).values

        # print(set(self.xgbt_model.feature_names)-set(X.columns.tolist()))

        return None
    
    def save_inferred(self, path_to_output):
        if self.inferred is None:
            print("inferred is None. No file was saved.")
        else:
            self.inferred.to_csv(path_to_output, index=False, header=False)
            print("inferred was successfully saved.")

In [4]:
class Pred_Attack(Attack_Di_Base):
    """
    予測が正解した行を優先しつつ、上位K件（または割合）だけを 1 にする版
    - topk:      ちょうどK件を1にする（推奨）
    - pos_ratio: 全体に対する割合で1件数を指定（topk未指定のときのみ有効）
    - threshold: 従来どおりの0.5丸め→正誤フラグ（topk/pos_ratio未指定時にフォールバック）
    """
    def __init__(self, path_to_xgboost_model_json, threshold=0.5, topk=None, pos_ratio=None):
        super().__init__(path_to_xgboost_model_json)
        self.threshold = float(threshold)
        self.topk = topk
        self.pos_ratio = pos_ratio

    def infer(self, path_to_Ai_csv):
        super().infer(path_to_Ai_csv)

        # 予測確率
        p = self.xgbt_model.predict(xgb.DMatrix(self.X))
        y = self.y

        # 従来方式（しきい値で丸め→正誤）へフォールバック
        if self.topk is None and self.pos_ratio is None:
            pred01 = (p >= self.threshold).astype(int)
            inferred = (pred01 == y).astype(int)
            self.inferred = pd.DataFrame(inferred)
            return self.inferred

        # 1) マージン（正解かつ自信が高いほど大）
        margin = 1.0 - np.abs(p - y)  # [0,1]

        # 2) 正解/不正解マスク
        correct_mask = ((p >= self.threshold).astype(int) == y)

        n = len(y)
        # 目標件数Kを決定
        if self.topk is not None:
            K = int(self.topk)
        else:
            r = float(self.pos_ratio)
            r = min(max(r, 0.0), 1.0)
            K = int(round(r * n))
        K = max(0, min(K, n))

        sel = np.zeros(n, dtype=bool)
        if K > 0:
            # 3) まず正解の中から margin 大きい順に min(K, #correct) 件
            idx_correct = np.flatnonzero(correct_mask)
            kc = min(K, idx_correct.size)
            if kc > 0:
                # margin大 → 昇順ではなく降順取りたいので -margin でargpartition
                part_c = np.argpartition(-margin[idx_correct], kc - 1)[:kc]
                sel[idx_correct[part_c]] = True

            # 4) 足りなければ不正解から補完
            rem = K - sel.sum()
            if rem > 0:
                idx_incorrect = np.flatnonzero(~correct_mask & ~sel)
                if idx_incorrect.size > 0:
                    ki = min(rem, idx_incorrect.size)
                    part_i = np.argpartition(-margin[idx_incorrect], ki - 1)[:ki]
                    sel[idx_incorrect[part_i]] = True

        self.inferred = pd.DataFrame(sel.astype(int))
        return self.inferred

In [5]:
class Conf_Attack(Attack_Di_Base):
    """
    モデルが確信を持って正答した行をmemberと推定する
    - threshold: 旧来どおりの |p - y| 閾値方式
    - topk:      |p - y| が小さい順にちょうど K 件 True にする（推奨）
    - pos_ratio: データ数に対する割合で指定（topk未指定のときのみ適用）
    """
    def __init__(self, path_to_xgboost_model_json, threshold=0.1, topk=None, pos_ratio=None, random_tie_break=False, seed=0):
        super().__init__(path_to_xgboost_model_json)
        self.threshold = float(threshold)
        self.topk = topk
        self.pos_ratio = pos_ratio
        self.random_tie_break = random_tie_break
        self.seed = seed

    def infer(self, path_to_Ai_csv):
        super().infer(path_to_Ai_csv)

        # 予測確率（0..1）
        pred = self.xgbt_model.predict(xgb.DMatrix(self.X))
        # スコア = |p - y|
        score = np.abs(pred - self.y)  # ndarray shape (n,)

        n = score.shape[0]

        # --- K件指定の優先ロジック ---
        k = None
        if self.topk is not None:
            k = int(self.topk)
        elif self.pos_ratio is not None:
            # 0..1の割合で上限下限をクリップ
            r = float(self.pos_ratio)
            r = min(max(r, 0.0), 1.0)
            k = int(round(r * n))

        if k is not None:
            k = max(0, min(k, n))  # 範囲ガード
            mask = np.zeros(n, dtype=bool)
            if k > 0:
                # 距離が小さい順にK件を取得（安くて速い）
                # 同値が多い場合でも厳密に「K件」を選ぶため、閾値ではなくインデックスで切る
                idx = np.argpartition(score, k - 1)[:k]

                if self.random_tie_break:
                    # 同値の中の選抜をランダム化したい場合のみ
                    rng = np.random.default_rng(self.seed)
                    # k件ぶんをシャッフル（順位に意味をもたせない）
                    rng.shuffle(idx)

                mask[idx] = True

            # DataFrame(1列)で返す（既存仕様に合わせる）
            self.inferred = pd.DataFrame(mask.astype(int))
            return self.inferred

        # --- 従来のthreshold方式（fallback） ---
        inferred = (score <= self.threshold)
        self.inferred = pd.DataFrame(inferred.astype(int))
        return self.inferred

In [7]:
# if __name__ == "__main__":
#     ap = argparse.ArgumentParser(description="")
#     ap.add_argument("model_json", help="trained model JSON (Booster.save_model)")
#     ap.add_argument("Ai_csv", help="Ai.csv to attack")
#     args = ap.parse_args()

#     attacker = Pred_Attack(args.model_json)
#     pred = attacker.infer(args.Ai_csv)
#     attacker.save_inferred("inferred_membership1.csv")

#     attacker = Conf_Attack(args.model_json)
#     pred = attacker.infer(args.Ai_csv)
#     attacker.save_inferred("inferred_membership2.csv")

In [6]:
id = "01"
model_json = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\D{id}.json"
Ai_csv = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\A{id}.csv"
out = "C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out"
out_pred = f"{out}\\inferred_membership1_{id}_ex.csv"
out_conf = f"{out}\\inferred_membership2_{id}_ex.csv"

In [7]:
attacker = Pred_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_pred)

attacker = Conf_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_conf)

inferred was successfully saved.
inferred was successfully saved.


In [13]:
# 変更後の Pred_Attack / Conf_Attack を使う前提
# - Pred_Attack(..., topk=K または pos_ratio=r)
# - Conf_Attack(..., topk=K または pos_ratio=r)
# - Attack_Di_Base.infer 内は reindex で feature_names を強制整列＆0埋め済み

LOGFILE = r"C:\Users\kikus\Documents\統数研\pwscup2025-scripts\out\log_attack_Di_ex.txt"
OUTDIR  = r"C:\Users\kikus\Documents\統数研\pwscup2025-scripts\out"
BASEDIR = r"C:\Users\kikus\Documents\統数研\pwscup2025-scripts\out\PWSCUP2025_Pre_Data_for_Attack"

# --- 件数コントロール設定 ---
TOPK_pred = None       # ちょうどこの件数を1にする（優先）
TOPK_conf = 20000       # ちょうどこの件数を1にする（優先）
POS_RATIO = None   # もしくは 0.10 など割合で指定（TOPKがNoneのときのみ適用）

def log_counts(id_str, path_csv, label):
    """0/1の件数を安定にログへ記録"""
    s = pd.read_csv(path_csv, header=None).iloc[:, 0]
    n1 = int((s == 1).sum())
    n0 = int((s == 0).sum())
    with open(LOGFILE, "a", encoding="utf-8") as f:
        print(f"{id_str} [{label}] -> ones={n1}, zeros={n0}, total={len(s)}", file=f)

# %%
# ID 01..20 を一括実行
for id_int in range(1, 21):
    id_str = f"{id_int:02d}"
    print(id_str)

    model_json = os.path.join(BASEDIR, f"D{id_str}.json")
    Ai_csv     = os.path.join(BASEDIR, f"A{id_str}.csv")
    out_pred   = os.path.join(OUTDIR,  f"inferred_membership1_{id_str}_ex.csv")
    out_conf   = os.path.join(OUTDIR,  f"inferred_membership2_{id_str}_ex.csv")

    try:
        # --- Pred（正解優先のトップK） ---
        attacker = Pred_Attack(model_json, topk=TOPK_pred, pos_ratio=POS_RATIO)
        _ = attacker.infer(Ai_csv)
        attacker.save_inferred(out_pred)
        log_counts(id_str, out_pred, "Pred(topK)")

        # --- Conf（|p−y|小のトップK） ---
        attacker = Conf_Attack(model_json, topk=TOPK_conf, pos_ratio=POS_RATIO)
        _ = attacker.infer(Ai_csv)
        attacker.save_inferred(out_conf)
        log_counts(id_str, out_conf, "Conf(topK)")

    except Exception as e:
        with open(LOGFILE, "a", encoding="utf-8") as f:
            print(f"{id_str} ERROR: {repr(e)}", file=f)

# %%
# つづけて ID=22 も実行
id_int = 22
id_str = f"{id_int:02d}"
print(id_str)

model_json = os.path.join(BASEDIR, f"D{id_str}.json")
Ai_csv     = os.path.join(BASEDIR, f"A{id_str}.csv")
out_pred   = os.path.join(OUTDIR,  f"inferred_membership1_{id_str}_ex.csv")
out_conf   = os.path.join(OUTDIR,  f"inferred_membership2_{id_str}_ex.csv")

try:
    attacker = Pred_Attack(model_json, topk=TOPK_pred, pos_ratio=POS_RATIO)
    _ = attacker.infer(Ai_csv)
    attacker.save_inferred(out_pred)
    log_counts(id_str, out_pred, "Pred(topK)")

    attacker = Conf_Attack(model_json, topk=TOPK_conf, pos_ratio=POS_RATIO)
    _ = attacker.infer(Ai_csv)
    attacker.save_inferred(out_conf)
    log_counts(id_str, out_conf, "Conf(topK)")

except Exception as e:
    with open(LOGFILE, "a", encoding="utf-8") as f:
        print(f"{id_str} ERROR: {repr(e)}", file=f)


01
inferred was successfully saved.
inferred was successfully saved.
02
inferred was successfully saved.
inferred was successfully saved.
03
inferred was successfully saved.
inferred was successfully saved.
04
inferred was successfully saved.
inferred was successfully saved.
05
inferred was successfully saved.
inferred was successfully saved.
06
inferred was successfully saved.
inferred was successfully saved.
07
inferred was successfully saved.
inferred was successfully saved.
08
inferred was successfully saved.
inferred was successfully saved.
09
inferred was successfully saved.
inferred was successfully saved.
10
inferred was successfully saved.
inferred was successfully saved.
11
inferred was successfully saved.
inferred was successfully saved.
12
inferred was successfully saved.
inferred was successfully saved.
13
inferred was successfully saved.
inferred was successfully saved.
14
inferred was successfully saved.
inferred was successfully saved.
15
inferred was successfully saved